# Índice de Evitación por Inseguridad (IEI): construcción y resultados
**EVPI 2023-2024 vs. 2025-2026 · OMSCGR**

Este cuaderno reproduce el informe técnico del IEI y el Excel `EVPI_indice_evitacion_inseguridad.xlsx`.

1. **Covariables.** Armonización de variables sociodemográficas, de arraigo, victimización, entorno y percepción.
2. **Ítems.** Codificación del módulo de cambios de hábitos (CH_1 a CH_6).
3. **Validación psicométrica.** Correlaciones tetracóricas, unidimensionalidad, α ordinal, KR-20, IRT 2PL y funcionamiento diferencial (DIF) por ola.
4. **Índice.** Versión principal y cuatro alternativas.
5. **Cambio entre olas.** Medias, distribución y subgrupos.
6. **Determinantes.** Modelos anidados, modelos por ola con prueba de Wald y descomposición Oaxaca-Blinder.
7. **Sensibilidad.**
8. **Figuras y exportación a Excel**, con el índice por persona.

**Definición.** Cada ítem vale 1 si la actividad *disminuyó* y 0 si *aumentó* o *se mantuvo*; "no aplica" y Ns/Nc quedan como faltantes. El IEI es el porcentaje de actividades aplicables que la persona redujo, para quienes responden al menos 3 de los 6 ítems:

$$\mathrm{IEI}_i = 100 \times \frac{\sum_{j \in A_i} \mathbb{1}[\text{disminuyó}_{ij}]}{|A_i|}, \qquad |A_i| \geq 3$$

Todas las estimaciones usan `peso_az_estrato` normalizado a media 1, con errores robustos HC1. Equivale a `regress … [pw=…], robust` en Stata.

## 0. Instalación, librerías y carga del archivo

In [ ]:
!pip install -q girth openpyxl

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import os, pickle
import pandas as pd, numpy as np
import statsmodels.api as sm
from scipy import stats, optimize
import girth
import matplotlib
import matplotlib.pyplot as plt
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 40)
SALIDA = 'salidas_iei'; os.makedirs(SALIDA, exist_ok=True)

In [ ]:
# Opción A (por defecto): subir el Excel. Opción B: montar Drive y poner la ruta en RUTA.
try:
    from google.colab import files
    EN_COLAB = True
except ImportError:
    EN_COLAB = False
RUTA = 'EVPI_2024_2026_VF_1.xlsx'
# from google.colab import drive; drive.mount('/content/drive'); RUTA = '/content/drive/MyDrive/.../EVPI_2024_2026_VF_1.xlsx'
if EN_COLAB and not os.path.exists(RUTA):
    RUTA = list(files.upload())[0]
d = pd.read_excel(RUTA).copy()
print(d.shape)

## 1. Covariables

Las covariables se armonizan entre olas y se agrupan en cuatro bloques:

- **Sociodemográficas:** sexo, edad, educación, NSE, ingreso del hogar, trabajo, discapacidad, nacionalidad y estado civil.
- **Arraigo y cohesión barrial.**
- **Victimización y entorno:** victimización, problemas del barrio, civiles armados y confianza en la Policía.
- **Percepción de inseguridad.**

Todos los modelos incluyen además efectos fijos de administración zonal, con Eloy Alfaro como referencia.

In [ ]:
W, Y, AZ = 'peso_az_estrato', 'AÑO', 'ADMINISTRACION ZONAL'

def col(code):
    m = [c for c in d.columns if c.split(' - ')[0].strip() == code and '?/' not in c]
    assert len(m) == 1, code
    return d[m[0]].astype(str).str.strip()

X = pd.DataFrame(index=d.index)
X['w'] = d[W] / d[W].mean()
X['ola25'] = (d[Y] == 2025).astype(int)
X['az'] = d[AZ]
# ---------- resultado ----------
ch6 = col('CH_6')
X['ch6_cat'] = ch6
X['y'] = np.where(ch6 == '3. Disminuyó', 1, np.where(ch6.isin(['1. Aumentó', '2. Se mantuvo']), 0, np.nan))
X['y_na1'] = np.where(ch6 == '5. No aplica', 1, X['y'])       # cota superior: No aplica = disminuyó
X['y_na0'] = np.where(ch6 == '5. No aplica', 0, X['y'])       # cota inferior: No aplica = no disminuyó
X['y_ord'] = ch6.map({'1. Aumentó': 0, '2. Se mantuvo': 1, '3. Disminuyó': 2})
# ---------- covariables ----------
X['mujer'] = (col('DD_1') == 'Mujer').astype(int)
edad = d['RANGO ETAREO'].astype(str)
for k, v in {'e26_40': '26 A 40', 'e41_55': '41 A 55', 'e56_64': '56 A 64', 'e65': '65 Y MÁS'}.items():
    X[k] = (edad == v).astype(int)                                  # ref: 18-25
ed = col('DD_7').str[0]
X['ed_sec'] = ed.isin(['D', 'E']).astype(int)                       # ref: ninguno/primaria
X['ed_tec'] = ed.isin(['F', 'G']).astype(int)
X['ed_uni'] = ed.isin(['H', 'I', 'J', 'K', 'L', 'M']).astype(int)
X['nse_medio'] = (d['nse3'] == 2).astype(int)                       # ref: bajo
X['nse_alto'] = (d['nse3'] == 3).astype(int)
ing = col('DD_10').str[0]
X['ing_460_919'] = (ing == 'B').astype(int)                         # ref: < $459
X['ing_920'] = ing.isin(list('CDEFGHIJ')).astype(int)
X['ing_nsnc'] = (ing == 'K').astype(int)
X['trabaja'] = (col('DD_8') == '1. Sí').astype(int)
X['discap'] = (col('DD_3') == '1. Sí').astype(int)
X['extranj'] = (col('DD_4') == '2. No').astype(int)
X['unido'] = col('DD_5').str[0].isin(['A', 'D']).astype(int)
anios = pd.to_numeric(col('SP_1'), errors='coerce')
X['anios_barrio10'] = anios.clip(upper=60) / 10
X['vecinos_mal'] = col('SP_2').str[0].isin(['3', '4']).astype(int)   # no se lleva / no los conoce
X['poco_orgullo'] = col('SP_3').str[0].isin(['3', '4']).astype(int)
X['no_quiere_quedarse'] = (col('SP_4') == '2. No').astype(int)
X['participa'] = (col('SP_5') == '1. Sí').astype(int)
vic = pd.concat([(col(k) == '1. Sí').astype(int) for k in ['VI_7', 'VI_8', 'VI_9', 'VI_11', 'VI_12', 'VI_13']], axis=1)
X['vic_robo_viol'] = (col('VI_6') == '1. Sí').astype(int)
X['vic_otro_pers'] = vic.max(axis=1)
X['vic_vivienda'] = (col('VI_4') == '1. Sí').astype(int)
X['incivil'] = pd.concat([(col(f'CC_{i}') == '1. Sí').astype(int) for i in range(1, 10)], axis=1).sum(axis=1)
X['armas'] = (col('TA_3') == '1. Sí').astype(int)
cp = col('CIE_4_1')
X['conf_pol_no'] = (cp == '2. No').astype(int)                       # ref: confía
X['conf_pol_desc'] = (~cp.isin(['1. Sí', '2. No'])).astype(int)      # no conoce funciones
psc2 = col('PSC_2')
X['inseg_barrio'] = (psc2 == '1. Muy Inseguro + Inseguro').astype(int)
X['inseg_barrio_na'] = (psc2 == '98. No Aplica').astype(int)
X['delinc_aumento'] = (col('PSC_16') == '1. Aumentó').astype(int)
X['inseg_noche'] = (col('PSC_15') == '1. Muy Inseguro + Inseguro').astype(int)
X['inseg_dia'] = (col('PSC_14') == '1. Muy Inseguro + Inseguro').astype(int)
X['sec'] = d['sec']

AZS = sorted(X.az.unique()); REF_AZ = 'ELOY ALFARO'
for z in AZS:
    if z != REF_AZ: X['az_' + z.replace(' ', '_')] = (X.az == z).astype(int)
AZD = [c for c in X if c.startswith('az_')]

BLOQUES = {
 'Sociodemográficas': ['mujer','e26_40','e41_55','e56_64','e65','ed_sec','ed_tec','ed_uni','nse_medio','nse_alto',
                       'ing_460_919','ing_920','ing_nsnc','trabaja','discap','extranj','unido'],
 'Arraigo y cohesión barrial': ['anios_barrio10','vecinos_mal','poco_orgullo','no_quiere_quedarse','participa'],
 'Victimización y entorno': ['vic_robo_viol','vic_otro_pers','vic_vivienda','incivil','armas','conf_pol_no','conf_pol_desc'],
 'Percepción de inseguridad': ['inseg_barrio','inseg_barrio_na','delinc_aumento'],
}
LAB = {
 'ola25':'Ola 2025-2026 (vs. 2023-2024)','mujer':'Mujer','e26_40':'Edad 26-40 (vs. 18-25)','e41_55':'Edad 41-55','e56_64':'Edad 56-64',
 'e65':'Edad 65 y más','ed_sec':'Secundaria (vs. primaria o menos)','ed_tec':'Técnica/tecnológica','ed_uni':'Universitaria o más',
 'nse_medio':'NSE medio (vs. bajo)','nse_alto':'NSE alto','ing_460_919':'Ingreso hogar $460-919 (vs. <$459)','ing_920':'Ingreso hogar ≥ $920',
 'ing_nsnc':'Ingreso hogar NS/NC','trabaja':'Trabaja','discap':'Tiene discapacidad','extranj':'Nacionalidad extranjera',
 'unido':'Casado/a o unión libre','anios_barrio10':'Años en el barrio (por 10 años)','vecinos_mal':'No se lleva / no conoce a vecinos',
 'poco_orgullo':'Poco o nada orgulloso/a del barrio','no_quiere_quedarse':'No quiere seguir viviendo en el barrio',
 'participa':'Participa en iniciativa barrial de seguridad','vic_robo_viol':'Víctima de robo con violencia','vic_otro_pers':'Víctima de otro delito personal',
 'vic_vivienda':'Robo a la vivienda','incivil':'Problemas en el barrio (índice 0-9, por problema)','armas':'Vio civiles armados en el barrio',
 'conf_pol_no':'No confía en la Policía (vs. confía)','conf_pol_desc':'No conoce funciones de la Policía','inseg_barrio':'Se siente inseguro/a en su barrio',
 'inseg_barrio_na':'Inseguridad en barrio: no aplica','delinc_aumento':'Cree que la delincuencia aumentó en su barrio',
 'inseg_noche':'Inseguro/a caminando de noche','inseg_dia':'Inseguro/a caminando de día'}
for z in AZS:
    if z != REF_AZ: LAB['az_' + z.replace(' ', '_')] = f'AZ {z.title()} (vs. Eloy Alfaro)'

GROUPS = {'edad': ['e26_40','e41_55','e56_64','e65'], 'educ': ['ed_sec','ed_tec','ed_uni'], 'nse': ['nse_medio','nse_alto'],
          'ing': ['ing_460_919','ing_920','ing_nsnc'], 'conf': ['conf_pol_no','conf_pol_desc'], 'insb': ['inseg_barrio','inseg_barrio_na'], 'az': AZD}
CONT = ['anios_barrio10', 'incivil']

def spec(k):
    v = ['ola25']
    for b in list(BLOQUES)[:k]: v += BLOQUES[b]
    return v + AZD

# ---------- estimación ----------

## 2. Ítems del módulo de cambios de hábitos

% que redujo cada actividad (entre quienes la actividad aplica), % que la aumentó y % de "no aplica" sobre el total, por ola.

In [ ]:
ITEMS = ['CH_1', 'CH_2', 'CH_3', 'CH_4', 'CH_5', 'CH_6']
ILAB = {'CH_1': 'Salir de noche', 'CH_2': 'Visitar espacios recreativos públicos', 'CH_3': 'Dejar salir solos a hijos menores',
        'CH_4': 'Visitar lugares turísticos', 'CH_5': 'Asistir a eventos culturales y deportivos', 'CH_6': 'Caminar por su barrio'}
B = pd.DataFrame(index=d.index); O_ = pd.DataFrame(index=d.index); NA = pd.DataFrame(index=d.index)
for k in ITEMS:
    s = col(k)
    B[k] = np.where(s == '3. Disminuyó', 1.0, np.where(s.isin(['1. Aumentó', '2. Se mantuvo']), 0.0, np.nan))
    O_[k] = s.map({'1. Aumentó': -1.0, '2. Se mantuvo': 0.0, '3. Disminuyó': 1.0})
    NA[k] = (s == '5. No aplica').astype(int)
w, ola = X.w, X.ola25
P = {}

# ---------- 1. Ítems ----------
rows = []
for k in ITEMS:
    for o in [0, 1]:
        m = (ola == o)
        mv = m & B[k].notna()
        rows.append(dict(item=k, ola=o, pct_red=np.average(B.loc[mv, k], weights=w[mv]),
                         pct_aum=np.average((O_.loc[mv, k] == -1), weights=w[mv]),
                         pct_na=np.average(NA.loc[m, k], weights=w[m]), n=int(mv.sum())))
P['items'] = pd.DataFrame(rows)


P['items'].round(3)

## 3. Validación psicométrica

Se calculan cuatro cosas:

- Correlaciones **tetracóricas** ponderadas, estimadas par a par por máxima verosimilitud.
- **Un factor** por ejes principales iterados, con autovalores, cargas y α ordinal.
- **KR-20** sobre casos completos de los 5 ítems sin CH_3, y la correlación ítem-resto.
- Comparación de cargas entre olas con la **congruencia de Tucker**.

In [ ]:
# ---------- 2. Psicometría ----------
def tetra(x, y, ww=None):
    m = ~(np.isnan(x) | np.isnan(y)); x, y = x[m], y[m]; ww = np.ones(len(x)) if ww is None else ww[m]
    n11 = ww[(x == 1) & (y == 1)].sum(); n10 = ww[(x == 1) & (y == 0)].sum(); n01 = ww[(x == 0) & (y == 1)].sum(); n00 = ww[(x == 0) & (y == 0)].sum()
    N = n11 + n10 + n01 + n00
    tx = stats.norm.ppf((n00 + n01) / N); ty = stats.norm.ppf((n00 + n10) / N)
    def nll(r):
        p00 = stats.multivariate_normal.cdf([tx, ty], cov=[[1, r], [r, 1]])
        px0 = stats.norm.cdf(tx); py0 = stats.norm.cdf(ty)
        p01 = px0 - p00; p10 = py0 - p00; p11 = 1 - px0 - py0 + p00
        ps = np.clip([p00, p01, p10, p11], 1e-12, 1)
        return -(n00*np.log(ps[0]) + n01*np.log(ps[1]) + n10*np.log(ps[2]) + n11*np.log(ps[3]))
    return optimize.minimize_scalar(nll, bounds=(-.99, .99), method='bounded').x
def tetra_mat(Bm, ww):
    k = Bm.shape[1]; M = np.eye(k)
    for i in range(k):
        for j in range(i + 1, k):
            M[i, j] = M[j, i] = tetra(Bm[:, i], Bm[:, j], ww)
    return M
def one_factor(M):
    # factorización de ejes principales iterada (1 factor)
    h = 1 - 1 / np.diag(np.linalg.inv(M))
    for _ in range(500):
        Mr = M.copy(); np.fill_diagonal(Mr, h)
        ev, vec = np.linalg.eigh(Mr); l = vec[:, -1] * np.sqrt(max(ev[-1], 0))
        hn = l**2
        if np.max(np.abs(hn - h)) < 1e-7: break
        h = hn
    l = l if l.sum() > 0 else -l
    return l, np.linalg.eigvalsh(M)[::-1]
psy = {}
for name, msk in [('Conjunto', np.ones(len(X), bool)), ('2023-2024', (ola == 0).values), ('2025-2026', (ola == 1).values)]:
    M = tetra_mat(B[ITEMS].values[msk], w.values[msk])
    lo, ev = one_factor(M)
    psy[name] = dict(tetra=pd.DataFrame(M, ITEMS, ITEMS), loadings=pd.Series(lo, ITEMS), eig=ev)
lo23, lo25 = psy['2023-2024']['loadings'], psy['2025-2026']['loadings']
P['tucker'] = float((lo23 @ lo25) / np.sqrt((lo23 @ lo23) * (lo25 @ lo25)))
P['psy'] = psy
# alfa ordinal (sobre tetracóricas) y KR-20 en casos completos de 5 ítems (sin CH_3)
def alpha_from_corr(M):
    k = M.shape[0]; return k / (k - 1) * (1 - k / M.sum())
P['alpha_ord'] = {k: alpha_from_corr(v['tetra'].values) for k, v in psy.items()}
core5 = ['CH_1', 'CH_2', 'CH_4', 'CH_5', 'CH_6']
cc = B[core5].dropna()
def kr20(df):
    k = df.shape[1]; p = df.mean(); return k / (k - 1) * (1 - (p*(1-p)).sum() / df.sum(axis=1).var(ddof=0))
P['kr20'] = {'Conjunto': kr20(cc), '2023-2024': kr20(B.loc[cc.index[ola[cc.index] == 0], core5]), '2025-2026': kr20(B.loc[cc.index[ola[cc.index] == 1], core5]), 'n': len(cc)}
# correlación ítem-resto (casos completos 5 ítems)
P['item_rest'] = {k: np.corrcoef(cc[k], cc.drop(columns=k).sum(axis=1))[0, 1] for k in core5}


print('Autovalores:', {k: np.round(v['eig'], 2) for k, v in P['psy'].items()})
print('α ordinal:', {k: round(v, 3) for k, v in P['alpha_ord'].items()}, '| KR-20:', {k: round(v, 3) for k, v in P['kr20'].items()})
print('Tucker:', round(P['tucker'], 3))
pd.DataFrame({k: v['loadings'] for k, v in P['psy'].items()}).round(3)

### IRT 2PL

El modelo logístico de dos parámetros se estima por máxima verosimilitud marginal con `girth`, sin pesos. Los faltantes ("no aplica", Ns/Nc) se tratan como respuestas no observadas.

In [ ]:
# ---------- 3. IRT 2PL ----------
def irt_fit(Bm):
    data = np.where(np.isnan(Bm), girth.INVALID_RESPONSE, Bm).astype(int).T
    est = girth.twopl_mml(data)
    return est, data
est_all, data_all = irt_fit(B[ITEMS].values)
P['irt'] = pd.DataFrame({'a': est_all['Discrimination'], 'b': est_all['Difficulty']}, index=ITEMS)
irt_w = {}
for o in [0, 1]:
    e, _ = irt_fit(B[ITEMS].values[(ola == o).values])
    irt_w[o] = pd.DataFrame({'a': e['Discrimination'], 'b': e['Difficulty']}, index=ITEMS)
P['irt_wave'] = irt_w
theta = girth.ability_eap(data_all, est_all['Difficulty'], est_all['Discrimination'])


P['irt'].round(3)

### Funcionamiento diferencial (DIF) por ola

Para cada ítem se estima un logit del ítem sobre el puntaje en los demás ítems y la ola (DIF uniforme), y luego sobre su interacción (DIF no uniforme). Un OR de ola distinto de 1 indica que, para un mismo nivel de evitación, el ítem se responde distinto en cada ola.

In [ ]:
# ---------- 4. DIF (uniforme y no uniforme) por ola ----------
import statsmodels.api as sm
dif = []
for k in ITEMS:
    rest = [j for j in ITEMS if j != k]
    rs = B[rest].mean(axis=1) * len(rest)              # puntaje resto (proporción × k)
    m = B[k].notna() & (B[rest].notna().sum(axis=1) >= 3)
    dd = pd.DataFrame({'y': B.loc[m, k], 'rest': B.loc[m, rest].mean(axis=1), 'ola': ola[m]})
    dd['int'] = dd.rest * dd.ola
    r1 = sm.GLM(dd.y, sm.add_constant(dd[['rest', 'ola']]), family=sm.families.Binomial(), var_weights=w[m]).fit(cov_type='HC1')
    r2 = sm.GLM(dd.y, sm.add_constant(dd[['rest', 'ola', 'int']]), family=sm.families.Binomial(), var_weights=w[m]).fit(cov_type='HC1')
    dif.append(dict(item=k, or_ola=np.exp(r1.params['ola']), p_unif=r1.pvalues['ola'], p_nounif=r2.pvalues['int'], n=int(m.sum())))
P['dif'] = pd.DataFrame(dif)


P['dif'].round(3)

## 4. Construcción del índice y versiones alternativas

| Versión | Definición |
|---|---|
| `iei` | Principal: 6 ítems, mínimo 3 aplicables, 0-100 |
| `iei_sin_ch6` | Sin caminar por el barrio (ítem con DIF) |
| `iei_core4` | CH_1, CH_2, CH_4 y CH_5 |
| `iei_ord` | Aumentó = −1, se mantuvo = 0, disminuyó = +1, reescalado a 0-100 |
| `iei_irt` | Puntaje EAP del 2PL, en desviaciones estándar |
| `alguna` | Redujo al menos una actividad (0/1) |

In [ ]:
# ---------- 5. Índices ----------
nval = B[ITEMS].notna().sum(axis=1)
IDX = pd.DataFrame(index=d.index)
IDX['n_items'] = nval
IDX['iei'] = np.where(nval >= 3, B[ITEMS].mean(axis=1) * 100, np.nan)                     # principal
core4 = ['CH_1', 'CH_2', 'CH_4', 'CH_5']
IDX['iei_core4'] = np.where(B[core4].notna().sum(axis=1) >= 3, B[core4].mean(axis=1) * 100, np.nan)
IDX['iei_sin_ch6'] = np.where(B[[k for k in ITEMS if k != 'CH_6']].notna().sum(axis=1) >= 3, B[[k for k in ITEMS if k != 'CH_6']].mean(axis=1) * 100, np.nan)
IDX['iei_ord'] = np.where(O_[ITEMS].notna().sum(axis=1) >= 3, (O_[ITEMS].mean(axis=1) + 1) / 2 * 100, np.nan)
IDX['iei_irt'] = np.where(nval >= 3, theta, np.nan)
# índice "alguna evitación"
IDX['alguna'] = np.where(nval >= 3, (B[ITEMS].max(axis=1)).astype(float), np.nan)
P['corr_idx'] = IDX[['iei', 'iei_core4', 'iei_sin_ch6', 'iei_ord', 'iei_irt']].corr()
P['cover'] = {o: dict(n=int((ola == o).sum()), con_indice=int(IDX.loc[ola == o, 'iei'].notna().sum()),
                       media_items=float(nval[ola == o].mean())) for o in [0, 1]}
IDX['ola25'] = ola; IDX['w'] = w
print(P['cover'])
P['corr_idx'].round(3)

## 5. Cambio entre olas: medias, distribución y subgrupos

In [ ]:
def SUBG_DEF(X):
    return {
 'Sexo': [('Hombre', X.mujer == 0), ('Mujer', X.mujer == 1)],
 'Edad': [('18-25', (X[['e26_40','e41_55','e56_64','e65']].sum(axis=1) == 0)), ('26-40', X.e26_40 == 1), ('41-55', X.e41_55 == 1), ('56-64', X.e56_64 == 1), ('65 y más', X.e65 == 1)],
 'Nivel socioeconómico': [('Bajo', (X.nse_medio + X.nse_alto) == 0), ('Medio', X.nse_medio == 1), ('Alto', X.nse_alto == 1)],
 'Educación': [('Primaria o menos', (X.ed_sec + X.ed_tec + X.ed_uni) == 0), ('Secundaria', X.ed_sec == 1), ('Técnica/tecnológica', X.ed_tec == 1), ('Universitaria o más', X.ed_uni == 1)],
 'Victimización personal (12 meses)': [('No víctima', (X.vic_robo_viol + X.vic_otro_pers) == 0), ('Víctima', (X.vic_robo_viol + X.vic_otro_pers) > 0)],
 'Problemas en el barrio': [('0-2 problemas', X.incivil <= 2), ('3-5 problemas', X.incivil.between(3, 5)), ('6-9 problemas', X.incivil >= 6)],
 'Percepción en el barrio': [('Se siente seguro/a', X.inseg_barrio == 0), ('Se siente inseguro/a', X.inseg_barrio == 1)],
 'Administración zonal': [(z.title().replace('Saenz', 'Sáenz').replace('Calderon', 'Calderón'), X.az == z) for z in AZS],
}

for c in ['iei', 'iei_core4', 'iei_sin_ch6', 'iei_ord', 'iei_irt', 'alguna', 'n_items']: X[c] = IDX[c]
R = {}

def wls(df, yv, xs, wv='w', cluster=None):
    m = df[[yv, wv] + xs].notna().all(axis=1); dd = df[m]
    mod = sm.WLS(dd[yv], sm.add_constant(dd[xs].astype(float)), weights=dd[wv])
    r = mod.fit(cov_type='HC1') if cluster is None else mod.fit(cov_type='cluster', cov_kwds={'groups': pd.factorize(dd[cluster])[0]})
    return r, dd

def change(df, yv, wv='w', cluster=None):
    r, dd = wls(df, yv, ['ola25'], wv, cluster)
    g0, g1 = dd[dd.ola25 == 0], dd[dd.ola25 == 1]
    return dict(m23=np.average(g0[yv], weights=g0[wv]), m25=np.average(g1[yv], weights=g1[wv]),
                dif=r.params['ola25'], se=r.bse['ola25'], p=r.pvalues['ola25'], n=int(r.nobs))

# 1. Cambio por versión del índice
VERS = {'iei': 'Principal: % de actividades aplicables reducidas (6 ítems, mín. 3)', 'iei_sin_ch6': 'Sin caminar por el barrio (CH_6)',
        'iei_core4': 'Núcleo de 4 ítems (CH_1, CH_2, CH_4, CH_5)', 'iei_ord': 'Ordinal: −1 aumentó, 0 se mantuvo, +1 disminuyó (0-100)',
        'iei_irt': 'Puntaje IRT 2PL (EAP, desviaciones estándar)', 'alguna': 'Redujo al menos una actividad (0/1)'}
R['change'] = pd.DataFrame([dict(version=k, desc=v, **change(X, k)) for k, v in VERS.items()])
X['w1'] = 1.0
extra = [dict(version='iei', desc='Principal, sin ponderar', **change(X, 'iei', 'w1')),
         dict(version='iei', desc='Principal, EE agrupados por sector censal', **change(X, 'iei', cluster='sec'))]
R['change'] = pd.concat([R['change'], pd.DataFrame(extra)], ignore_index=True)

# distribución del índice por ola (bins)
bins = [-.1, 0.001, 25, 50, 75, 99.999, 100]; labs = ['0 (ninguna)', '1-25', '26-50', '51-75', '76-99', '100 (todas)']
dist = []
for o in [0, 1]:
    g = X[(X.ola25 == o) & X.iei.notna()]
    cat = pd.cut(g.iei, bins=bins, labels=labs)
    for l in labs: dist.append(dict(ola=o, tramo=l, pct=np.average(cat == l, weights=g.w)))
R['dist'] = pd.DataFrame(dist)

# 2. Subgrupos
rows = []
for grp, cats in SUBG_DEF(X).items():
    for lab, m in cats:
        c = change(X[m], 'iei'); rows.append(dict(grupo=grp, categoria=lab, **c))
R['subg'] = pd.DataFrame(rows)


R['change'].round(3)

In [ ]:
R['subg'].round(2)

## 6. Determinantes del índice

### 6.1 Modelos anidados

MCO ponderado con errores HC1. Se ajustan cuatro modelos que van sumando bloques de covariables:

1. **M1:** sociodemográficas.
2. **M2:** M1 + arraigo y cohesión barrial.
3. **M3:** M2 + victimización y entorno.
4. **M4:** M3 + percepción de inseguridad.

Todos incluyen la ola y efectos fijos de administración zonal.

In [ ]:
# 3. Determinantes (MCO ponderado sobre índice 0-100)
models = {}
for k, name in enumerate(['M1 Sociodemográficas', 'M2 + Arraigo y cohesión', 'M3 + Victimización y entorno', 'M4 + Percepción'], 1):
    r, dd = wls(X, 'iei', spec(k))
    models[name] = dict(params=r.params, bse=r.bse, p=r.pvalues, n=int(r.nobs), r2=r.rsquared)
R['models'] = models
xs4 = spec(4); xs4w = [v for v in xs4 if v != 'ola25']
by = {}
for o in [0, 1]:
    r, dd = wls(X[X.ola25 == o], 'iei', xs4w); by[o] = pd.DataFrame({'b': r.params, 'se': r.bse, 'p': r.pvalues})
cmp_ = by[0].join(by[1], lsuffix='_23', rsuffix='_25').drop('const')
cmp_['dif'] = cmp_.b_25 - cmp_.b_23; cmp_['se_dif'] = np.sqrt(cmp_.se_23**2 + cmp_.se_25**2)
cmp_['p_dif'] = 2 * (1 - stats.norm.cdf(np.abs(cmp_.dif / cmp_.se_dif)))
R['by_wave'] = cmp_
Xi = X.copy(); inter = []
for v in xs4w: Xi['x_' + v] = Xi[v] * Xi.ola25; inter.append('x_' + v)
r, dd = wls(Xi, 'iei', ['ola25'] + xs4w + inter)
for nm, sel in [('wald', inter), ('wald_noaz', ['x_' + v for v in xs4w if v not in AZD])]:
    t = r.wald_test(' = 0, '.join(sel) + ' = 0', scalar=True)
    R[nm] = dict(chi2=float(t.statistic), df=len(sel), p=float(t.pvalue))


for k, v in R['models'].items(): print(f"{k}: n = {v['n']}, R² = {v['r2']:.3f}, ola = {v['params']['ola25']:.2f}")
print('Wald (sin AZ):', R['wald_noaz'])
M4 = R['models']['M4 + Percepción']
pd.DataFrame({'b': M4['params'], 'EE': M4['bse'], 'p': M4['p']}).rename(index=LAB).round(3)

In [ ]:
R['by_wave'][['b_23', 'p_23', 'b_25', 'p_25', 'dif', 'p_dif']].rename(index=LAB).round(3)

### 6.2 Descomposición Oaxaca-Blinder

La descomposición tiene dos partes, explicada y no explicada, con los coeficientes del modelo conjunto con indicador de ola. Los intervalos se obtienen por bootstrap estratificado por ola, con 300 réplicas. El bootstrap tarda alrededor de 1 minuto.

In [ ]:
# 4. Oaxaca-Blinder
def oaxaca(df, xs, blocks, yv='iei'):
    df = df[df[yv].notna()]; g0, g1 = df[df.ola25 == 0], df[df.ola25 == 1]
    m0 = np.average(g0[xs], axis=0, weights=g0.w); m1 = np.average(g1[xs], axis=0, weights=g1.w)
    bp = sm.WLS(df[yv], sm.add_constant(df[['ola25'] + xs].astype(float)), weights=df.w).fit().params[xs].values
    y0 = np.average(g0[yv], weights=g0.w); y1 = np.average(g1[yv], weights=g1.w)
    c = (m1 - m0) * bp
    res = {'total': y1 - y0, 'explicado': c.sum(), 'no_explicado': (y1 - y0) - c.sum()}
    for b, vs in blocks.items(): res['E: ' + b] = sum(c[xs.index(v)] for v in vs if v in xs)
    return res
blocks = {**BLOQUES, 'Administración zonal': AZD}
pt = oaxaca(X, xs4w, blocks)
rng = np.random.default_rng(2026); base = X[X.iei.notna()]
bt = pd.DataFrame([oaxaca(pd.concat([g.sample(len(g), replace=True, random_state=int(rng.integers(1e9))) for _, g in base.groupby('ola25')]), xs4w, blocks) for _ in range(300)])
R['oaxaca'] = pd.DataFrame({'componente': list(pt), 'estimado': list(pt.values()), 'se': [bt[k].std() for k in pt],
                            'lo': [bt[k].quantile(.025) for k in pt], 'hi': [bt[k].quantile(.975) for k in pt]})
pt2 = oaxaca(X, xs4w, blocks, 'iei_sin_ch6')
R['oaxaca_sin_ch6'] = pt2


print('Sin CH_6:', {k: round(v, 2) for k, v in R['oaxaca_sin_ch6'].items()})
R['oaxaca'].round(2)

## 7. Sensibilidad del efecto de la ola (M4)

In [ ]:
# 5. Sensibilidad del efecto de ola condicional
sens = []
for yv, desc in [('iei', 'Principal'), ('iei_sin_ch6', 'Sin CH_6'), ('iei_core4', 'Núcleo 4 ítems'), ('iei_ord', 'Ordinal'), ('iei_irt', 'IRT (DE)')]:
    r, dd = wls(X, yv, xs4); sens.append(dict(especificacion=f'{desc}, M4', ola=r.params['ola25'], se=r.bse['ola25'], p=r.pvalues['ola25'], n=int(r.nobs)))
r, dd = wls(X, 'iei', xs4, cluster='sec'); sens.append(dict(especificacion='Principal, M4, EE agrupados por sector', ola=r.params['ola25'], se=r.bse['ola25'], p=r.pvalues['ola25'], n=int(r.nobs)))
r, dd = wls(X, 'iei', xs4, wv='w1'); sens.append(dict(especificacion='Principal, M4, sin ponderar', ola=r.params['ola25'], se=r.bse['ola25'], p=r.pvalues['ola25'], n=int(r.nobs)))
r, dd = wls(X, 'iei', xs4 + ['inseg_dia', 'inseg_noche']); sens.append(dict(especificacion='Principal, M4 + inseguridad al caminar día/noche', ola=r.params['ola25'], se=r.bse['ola25'], p=r.pvalues['ola25'], n=int(r.nobs)))
X['todos6'] = X.n_items == 6
r, dd = wls(X[X.todos6], 'iei', xs4); sens.append(dict(especificacion='Principal, M4, solo quienes responden los 6 ítems', ola=r.params['ola25'], se=r.bse['ola25'], p=r.pvalues['ola25'], n=int(r.nobs)))
R['sens'] = pd.DataFrame(sens)

R['sens'].round(3)

## 8. Figuras

In [ ]:
import pickle, pandas as pd, numpy as np, matplotlib
NAVY, BLUE, CYAN, GREEN, ORANGE, AMBER, GREY = '#15206b', '#4e59aa', '#1d9ebe', '#0fa27c', '#ff6840', '#ffb500', '#94a3b8'
plt.rcParams.update({'text.parse_math': False, 'font.family': 'DejaVu Sans', 'font.size': 9, 'axes.spines.top': False, 'axes.spines.right': False})
def es(v, d=1): return f'{v:.{d}f}'.replace('.', ',').replace('-', '−')
SRC = 'Fuente: EVPI 2023-2024 y 2025-2026, OMSCGR. Ponderado con peso_az_estrato.'
ILAB = {'CH_1': 'Salir de noche', 'CH_2': 'Visitar espacios\nrecreativos públicos', 'CH_3': 'Dejar salir solos\na hijos menores',
        'CH_4': 'Visitar lugares\nturísticos', 'CH_5': 'Asistir a eventos\nculturales/deportivos', 'CH_6': 'Caminar por\nsu barrio'}
ITEMS = list(ILAB)

# Fig A: ítems y psicometría
fig, axs = plt.subplots(1, 2, figsize=(13.5, 4.9), gridspec_kw={'width_ratios': [1.45, 1]})
ax = axs[0]; I = P['items']; x = np.arange(6)
for o, c, dx in [(0, BLUE, -.2), (1, ORANGE, .2)]:
    v = I[I.ola == o].set_index('item').loc[ITEMS]
    ax.bar(x + dx, v.pct_red*100, width=.38, color=c, label='2023-2024' if o == 0 else '2025-2026')
    for i, (val, na) in enumerate(zip(v.pct_red, v.pct_na)):
        ax.text(i + dx, val*100 + 1, es(val*100, 0), ha='center', fontsize=7.5, color=NAVY)
        ax.text(i + dx, 3, f'NA\n{es(na*100, 0)}%', ha='center', fontsize=6.3, color='white')
ax.set_xticks(x); ax.set_xticklabels([ILAB[k] for k in ITEMS], fontsize=7.2); ax.set_ylim(0, 95)
ax.set_ylabel('% que redujo (entre quienes la actividad aplica)'); ax.legend(frameon=False, fontsize=8, loc='upper right', ncol=2)
ax.set_title('Actividades reducidas por inseguridad, por ola\n(NA = % "no aplica" sobre el total)', loc='left', fontsize=10.5, color=NAVY, fontweight='bold')
ax = axs[1]
L = pd.DataFrame({k: P['psy'][k]['loadings'] for k in ['2023-2024', '2025-2026']}).loc[ITEMS]
y = np.arange(6)
ax.barh(y - .2, L['2023-2024'], height=.38, color=BLUE, label='2023-2024'); ax.barh(y + .2, L['2025-2026'], height=.38, color=ORANGE, label='2025-2026')
ax.set_yticks(y); ax.set_yticklabels([ILAB[k].replace('\n', ' ') for k in ITEMS], fontsize=8); ax.invert_yaxis(); ax.set_xlim(0, 1)
ax.set_xlabel('Carga factorial (1 factor, correlaciones tetracóricas)')
ev = P['psy']['Conjunto']['eig']
ax.set_title(f"Unidimensionalidad e invarianza entre olas\nAutovalores: {es(ev[0], 2)} / {es(ev[1], 2)} · α ordinal = {es(P['alpha_ord']['Conjunto'], 2)}\nCongruencia de Tucker = {es(P['tucker'], 3)}",
             loc='left', fontsize=10.5, color=NAVY, fontweight='bold')
ax.legend(frameon=False, fontsize=8, loc='lower right')
fig.text(.01, .01, SRC, fontsize=7.5, color='#64748b'); plt.tight_layout(rect=(0, .04, 1, 1))
plt.savefig(f'{SALIDA}/iei_fig1_items.png', dpi=170); plt.show()

# Fig B: distribución + cambio por subgrupos clave y AZ
fig, axs = plt.subplots(1, 2, figsize=(12, 5.2), gridspec_kw={'width_ratios': [1, 1.2]})
ax = axs[0]; D = R['dist']; labs = list(dict.fromkeys(D.tramo)); x = np.arange(len(labs))
for o, c, dx in [(0, BLUE, -.2), (1, ORANGE, .2)]:
    v = D[D.ola == o].set_index('tramo').loc[labs].pct*100
    ax.bar(x + dx, v, width=.38, color=c, label='2023-2024' if o == 0 else '2025-2026')
    for i, val in enumerate(v): ax.text(i + dx, val + .6, es(val, 0), ha='center', fontsize=7.5, color=NAVY)
ch = R['change'].iloc[0]
ax.set_xticks(x); ax.set_xticklabels(labs, fontsize=8); ax.set_xlabel('Índice de evitación (0-100)'); ax.set_ylabel('% de personas')
ax.legend(frameon=False, fontsize=8)
ax.set_title(f"Distribución del índice\nMedia: {es(ch.m23)} → {es(ch.m25)} ({es(ch.dif)} puntos, p < 0,001)", loc='left', fontsize=10.5, color=NAVY, fontweight='bold')
ax = axs[1]; S = R['subg']
keep = [('Victimización personal (12 meses)', None), ('Problemas en el barrio', None), ('Administración zonal', None)]
rows = []
for g, _ in keep:
    rows.append(('__' + g, None))
    sub = S[S.grupo == g]
    if g == 'Administración zonal': sub = sub.sort_values('dif')
    rows += [(r.categoria, r) for r in sub.itertuples()]
yy = 0; ticks, tl = [], []
for lab, r in rows:
    if r is None:
        ax.text(-30, yy, lab[2:], fontsize=8.5, fontweight='bold', color=NAVY, va='center'); yy += 1; continue
    c = (ORANGE if r.dif > 0 else BLUE) if r.p < .05 else GREY
    ax.plot([r.dif - 1.96*r.se, r.dif + 1.96*r.se], [yy, yy], color=c, lw=1.6)
    ax.scatter(r.dif, yy, color=c, s=30, zorder=3); ticks.append(yy); tl.append(lab); yy += 1
ax.axvline(0, color='#555', lw=.8, ls='--'); ax.axvline(ch.dif, color=NAVY, lw=.8, ls=':')
ax.text(ch.dif, -1.3, 'Media DMQ', ha='center', fontsize=7.5, color=NAVY)
ax.set_yticks(ticks); ax.set_yticklabels(tl, fontsize=8); ax.set_ylim(yy - .5, -2); ax.set_xlim(-30, 12)
ax.set_xlabel('Cambio en el índice 2023-2024 → 2025-2026 (puntos), IC 95%')
ax.set_title('Cambio del índice por grupo\n(la caída se concentra donde hay menos victimización y desorden)', loc='left', fontsize=10.5, color=NAVY, fontweight='bold')
fig.text(.01, .01, SRC + ' Gris: p ≥ 0,05.', fontsize=7.5, color='#64748b'); plt.tight_layout(rect=(0, .04, 1, 1))
plt.savefig(f'{SALIDA}/iei_fig2_cambio.png', dpi=170); plt.show()

# Fig C: determinantes (conjunto y por ola)
M4 = R['models']['M4 + Percepción']; Bw = R['by_wave']
sel = [v for v in sum(BLOQUES.values(), []) if v != 'inseg_barrio_na']
fig, axs = plt.subplots(1, 2, figsize=(13, 10.5), sharey=True)
for ax, mode in zip(axs, ['pool', 'wave']):
    for i, v in enumerate(sel):
        if mode == 'pool':
            b, se, p = M4['params'][v], M4['bse'][v], M4['p'][v]
            c = (ORANGE if b > 0 else BLUE) if p < .05 else GREY
            ax.plot([b - 1.96*se, b + 1.96*se], [i, i], color=c, lw=1.6); ax.scatter(b, i, color=c, s=30 if p < .05 else 16, zorder=3)
        else:
            for o, c, dy in [('23', BLUE, -.17), ('25', ORANGE, .17)]:
                b, se, p = Bw.loc[v, 'b_' + o], Bw.loc[v, 'se_' + o], Bw.loc[v, 'p_' + o]
                ax.plot([b - 1.96*se, b + 1.96*se], [i + dy]*2, color=c, lw=1.3, alpha=1 if p < .05 else .4)
                ax.scatter(b, i + dy, color=c, s=24, alpha=1 if p < .05 else .4, zorder=3, label=('2023-2024' if o == '23' else '2025-2026') if i == 0 else None)
            if Bw.loc[v, 'p_dif'] < .05: ax.text(24, i, '◆', color=NAVY, fontsize=9, va='center', ha='center')
    ax.axvline(0, color='#555', lw=.8, ls='--'); ax.set_xlim(-16, 26)
    ax.set_xlabel('Efecto sobre el índice (puntos de 0-100), IC 95%')
axs[0].set_yticks(range(len(sel))); axs[0].set_yticklabels([LAB[v] for v in sel], fontsize=8); axs[0].invert_yaxis()
axs[0].set_title(f"Determinantes del índice (ambas olas)\nMCO ponderado con EF de ola y AZ · R² = {es(M4['r2'], 3)}", loc='left', fontsize=10.5, color=NAVY, fontweight='bold')
W = R['wald_noaz']
axs[1].set_title(f"Por ola · ◆ diferencia significativa (p < 0,05)\nPrueba conjunta: χ²({W['df']}) = {es(W['chi2'])}, p < 0,001", loc='left', fontsize=10.5, color=NAVY, fontweight='bold')
axs[1].legend(frameon=False, fontsize=8.5, loc='lower right')
fig.text(.01, .005, SRC + ' Naranja: más evitación; azul: menos; gris/tenue: p ≥ 0,05.', fontsize=7.5, color='#64748b')
plt.tight_layout(rect=(0, .015, 1, 1)); plt.savefig(f'{SALIDA}/iei_fig3_determinantes.png', dpi=170); plt.show()

## 9. Exportación a Excel

El archivo replica el Excel del informe. Tiene notas, ítems, psicometría, índice y cambio, subgrupos, modelos anidados, determinantes por ola, descomposición, sensibilidad y figuras. Incluye además el **índice por persona**, con `ID_Todo`, para unirlo a la base.

In [ ]:
import pickle, pandas as pd, numpy as np
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.drawing.image import Image as XLImage
from openpyxl.utils import get_column_letter as L
from openpyxl.formatting.rule import FormulaRule
F = 'Arial'; fnt = lambda **k: Font(name=F, **k)
HDR = PatternFill('solid', fgColor='1F4E79'); BLK = PatternFill('solid', fgColor='D9E2F3'); BIN = '0000FF'
ITEMS = ['CH_1', 'CH_2', 'CH_3', 'CH_4', 'CH_5', 'CH_6']
ILAB = {'CH_1': 'Salir de noche', 'CH_2': 'Visitar espacios recreativos públicos', 'CH_3': 'Dejar salir solos a hijos menores',
        'CH_4': 'Visitar lugares turísticos', 'CH_5': 'Asistir a eventos culturales y deportivos', 'CH_6': 'Caminar por su barrio'}
def head(ws, r, hs, h=30):
    for j, t in enumerate(hs, 1):
        c = ws.cell(r, j, t); c.font = fnt(bold=True, color='FFFFFF', size=9); c.fill = HDR
        c.alignment = Alignment(wrap_text=True, horizontal='center', vertical='center')
    ws.row_dimensions[r].height = h
def title(ws, t, sub=None):
    ws['A1'] = t; ws['A1'].font = fnt(bold=True, size=13)
    if sub: ws['A2'] = sub; ws['A2'].font = fnt(italic=True, size=9, color='555555')
def widths(ws, w):
    for j, x in enumerate(w, 1): ws.column_dimensions[L(j)].width = x
def star(p): return '***' if p < .001 else '**' if p < .01 else '*' if p < .05 else '†' if p < .10 else ''
def put(ws, r, vals, fmts={}, blue=()):
    for j, v in enumerate(vals, 1):
        c = ws.cell(r, j, v); c.font = fnt(size=9, color=BIN if j in blue else '000000')
        if j in fmts: c.number_format = fmts[j]
def es(v, d=1): return f'{v:.{d}f}'.replace('.', ',').replace('-', '−')

ch = R['change'].set_index('version'); ob = R['oaxaca'].set_index('componente'); M4 = R['models']['M4 + Percepción']; Bw = R['by_wave']
S = R['subg'].set_index(['grupo', 'categoria'])
wb = Workbook()
ws = wb.active; ws.title = 'Notas'
c0 = R['change'].iloc[0]
notas = [
 ('Índice de Evitación por Inseguridad (IEI). Módulo de cambios de hábitos (CH_1 a CH_6) de la EVPI, 2023-2024 vs. 2025-2026', 'h'), ('', ''),
 ('Construcción', 'b'),
 ('Ítems: cambio en la frecuencia con la que la persona (1) sale de noche, (2) visita espacios recreativos públicos, (3) deja salir solos a sus hijos menores, (4) visita lugares turísticos, (5) asiste a eventos culturales y deportivos, (6) camina por su barrio.', ''),
 ('Cada ítem vale 1 si la actividad "disminuyó" y 0 si "aumentó" o "se mantuvo". "No aplica" y Ns/Nc quedan como faltantes: la persona no realiza la actividad o no tiene hijos menores.', ''),
 ('IEI = porcentaje de las actividades aplicables que la persona redujo por inseguridad (0-100). Se calcula para quienes responden al menos 3 de los 6 ítems.', ''),
 (f"Cobertura: {P['cover'][0]['con_indice']:,} de {P['cover'][0]['n']:,} casos en 2023-2024 y {P['cover'][1]['con_indice']:,} de {P['cover'][1]['n']:,} en 2025-2026. Ítems aplicables promedio: {es(P['cover'][0]['media_items'])} y {es(P['cover'][1]['media_items'])}.".replace(',', '.'), ''),
 ('', ''),
 ('Validez y confiabilidad', 'b'),
 (f"Unidimensionalidad: en la matriz de correlaciones tetracóricas el primer autovalor es {es(P['psy']['Conjunto']['eig'][0], 2)} y el segundo {es(P['psy']['Conjunto']['eig'][1], 2)}. Las cargas en un factor van de {es(P['psy']['Conjunto']['loadings'].min(), 2)} a {es(P['psy']['Conjunto']['loadings'].max(), 2)}.", ''),
 (f"Confiabilidad: α ordinal = {es(P['alpha_ord']['Conjunto'], 2)} (2023-2024: {es(P['alpha_ord']['2023-2024'], 2)}; 2025-2026: {es(P['alpha_ord']['2025-2026'], 2)}). KR-20 en casos completos de los 5 ítems sin CH_3 = {es(P['kr20']['Conjunto'], 2)}.", ''),
 (f"Invarianza entre olas: la congruencia de Tucker entre las cargas de ambas olas es {es(P['tucker'], 3)}, por lo que la estructura es la misma. El análisis de funcionamiento diferencial (DIF) muestra que CH_6 (caminar por su barrio) cae más que los otros ítems para un mismo nivel de evitación (OR = {es(P['dif'].set_index('item').loc['CH_6', 'or_ola'], 2)}, p < 0,001). Coincide con el cambio en \"no aplica\" de ese ítem (16% → 2%). Por eso se reportan versiones del índice sin CH_6.", ''),
 ('Versiones: principal (6 ítems); sin CH_6; núcleo de 4 ítems (CH_1, CH_2, CH_4, CH_5); ordinal (−1, 0, +1); puntaje IRT 2PL (EAP). Todas se correlacionan entre 0,91 y 0,99 con la principal.', ''),
 ('', ''),
 ('Hallazgos principales', 'b'),
 (f"1. El IEI baja de {es(c0.m23)} a {es(c0.m25)} ({es(c0.dif)} puntos, p < 0,001). Sin CH_6 la caída es de {es(abs(ch.loc['iei_sin_ch6', 'dif']))} puntos y con el núcleo de 4 ítems de {es(abs(ch.loc['iei_core4', 'dif']))}: la reducción es real, pero CH_6 la amplifica.", ''),
 (f"2. La evitación sigue siendo masiva: el {es(ch.loc['alguna', 'm25']*100)}% redujo al menos una actividad en 2025-2026 (frente al {es(ch.loc['alguna', 'm23']*100)}% en 2023-2024).", ''),
 (f"3. La composición de la muestra no explica el cambio: {es(ob.loc['explicado', 'estimado'], 2)} de {es(ob.loc['total', 'estimado'], 2)} puntos (Oaxaca-Blinder). Controlando por todas las covariables, el efecto de la ola es {es(R['sens'].iloc[0].ola)} puntos.", ''),
 (f"4. La caída es desigual. Entre quienes no fueron víctimas el IEI baja {es(abs(S.loc[('Victimización personal (12 meses)', 'No víctima'), 'dif']))} puntos; entre las víctimas, {es(abs(S.loc[('Victimización personal (12 meses)', 'Víctima'), 'dif']))} (n.s.). En barrios con 0-2 problemas baja {es(abs(S.loc[('Problemas en el barrio', '0-2 problemas'), 'dif']))} puntos y con 6-9 problemas {es(abs(S.loc[('Problemas en el barrio', '6-9 problemas'), 'dif']))} (n.s.). Los Chillos, Tumbaco, Choco Andino y Quitumbe no muestran una caída significativa.", ''),
 (f"5. Determinantes (modelo conjunto, R² = {es(M4['r2'], 3)}): sentirse inseguro/a en el barrio (+{es(M4['params']['inseg_barrio'])}), creer que la delincuencia aumentó (+{es(M4['params']['delinc_aumento'])}), no confiar en la Policía (+{es(M4['params']['conf_pol_no'])}), ser mujer (+{es(M4['params']['mujer'])}), edad de 41 años o más y NSE alto. La victimización directa no aumenta la evitación una vez controlada la percepción.", ''),
 (f"6. Los determinantes cambian entre olas (χ²({R['wald_noaz']['df']}) = {es(R['wald_noaz']['chi2'])}, p < 0,001). En 2025-2026 pesan más el desorden del barrio y ser mujer; deja de importar no conocer a la Policía; desaparece la menor evitación de los hogares de mayor ingreso.", ''),
 ('', ''),
 ('Especificación y límites', 'b'),
 ('Ponderación con peso_az_estrato. MCO ponderado con errores robustos HC1 (equivalente a regress [pw], robust). Descomposición Oaxaca-Blinder con coeficientes del modelo conjunto e IC bootstrap (300 réplicas estratificadas por ola).', ''),
 ('Las preguntas miden cambio en la frecuencia (flujo), no el nivel de actividad (stock). Una caída del IEI indica que menos personas reportan nuevas reducciones, no necesariamente que recuperaron sus actividades.', ''),
 ('La base de cada ítem depende de "no aplica", que bajó en todos los ítems entre olas (por ejemplo, eventos 35% → 26%). Las versiones alternativas y la muestra con los 6 ítems respondidos acotan este problema.', ''),
 ('Asociaciones, no efectos causales: la percepción de inseguridad y la evitación pueden reforzarse mutuamente.', ''),
 ('', ''),
 ('Fuente: EVPI 2023-2024 y 2025-2026, OMSCGR. Cálculos propios en Python (statsmodels, girth). *** p<0,001; ** p<0,01; * p<0,05; † p<0,10.', ''),
]
for i, (t, s) in enumerate(notas, 1):
    c = ws.cell(i, 1, t); c.alignment = Alignment(wrap_text=True, vertical='top'); c.font = fnt(bold=s in ('h', 'b'), size=13 if s == 'h' else 10)
ws.column_dimensions['A'].width = 150

# Ítems
ws = wb.create_sheet('Ítems'); title(ws, 'Actividades reducidas por inseguridad, por ítem y ola', '% que redujo entre quienes la actividad aplica; % no aplica sobre el total. Ponderado.')
head(ws, 4, ['Código', 'Actividad', '% redujo 2023', '% redujo 2025', 'Cambio (pp)', '% aumentó 2023', '% aumentó 2025', '% no aplica 2023', '% no aplica 2025', 'n 2023', 'n 2025'])
I = P['items']
for i, k in enumerate(ITEMS, 5):
    a = I[(I.item == k) & (I.ola == 0)].iloc[0]; b = I[(I.item == k) & (I.ola == 1)].iloc[0]
    put(ws, i, [k, ILAB[k], a.pct_red, b.pct_red, f'=(D{i}-C{i})*100', a.pct_aum, b.pct_aum, a.pct_na, b.pct_na, a.n, b.n],
        {3: '0.0%', 4: '0.0%', 5: '+0.0;-0.0', 6: '0.0%', 7: '0.0%', 8: '0.0%', 9: '0.0%', 10: '#,##0', 11: '#,##0'}, blue=(3, 4, 6, 7, 8, 9, 10, 11))
widths(ws, [8, 40, 11, 11, 11, 11, 11, 12, 12, 8, 8])

# Psicometría
ws = wb.create_sheet('Psicometría'); title(ws, 'Validez y confiabilidad del índice')
r = 3; ws.cell(r, 1, 'Cargas factoriales (1 factor, ejes principales sobre correlaciones tetracóricas ponderadas)').font = fnt(bold=True, size=10); r += 1
head(ws, r, ['Código', 'Actividad', 'Conjunto', '2023-2024', '2025-2026', 'Correlación ítem-resto', 'IRT a (discriminación)', 'IRT b (dificultad)', 'a 2023', 'a 2025', 'b 2023', 'b 2025']); r += 1
for k in ITEMS:
    put(ws, r, [k, ILAB[k], P['psy']['Conjunto']['loadings'][k], P['psy']['2023-2024']['loadings'][k], P['psy']['2025-2026']['loadings'][k],
                P['item_rest'].get(k, None), P['irt'].loc[k, 'a'], P['irt'].loc[k, 'b'], P['irt_wave'][0].loc[k, 'a'], P['irt_wave'][1].loc[k, 'a'],
                P['irt_wave'][0].loc[k, 'b'], P['irt_wave'][1].loc[k, 'b']], {j: '0.000' for j in range(3, 13)}, blue=range(3, 13)); r += 1
r += 1
for lab_, v in [('Primer autovalor (conjunto)', P['psy']['Conjunto']['eig'][0]), ('Segundo autovalor (conjunto)', P['psy']['Conjunto']['eig'][1]),
                ('α ordinal conjunto', P['alpha_ord']['Conjunto']), ('α ordinal 2023-2024', P['alpha_ord']['2023-2024']), ('α ordinal 2025-2026', P['alpha_ord']['2025-2026']),
                ('KR-20 (5 ítems sin CH_3, casos completos)', P['kr20']['Conjunto']), ('Congruencia de Tucker entre olas', P['tucker'])]:
    ws.cell(r, 1, lab_).font = fnt(size=9); c = ws.cell(r, 3, v); c.number_format = '0.000'; c.font = fnt(size=9, color=BIN); r += 1
r += 1; ws.cell(r, 1, 'Funcionamiento diferencial por ola (DIF): logit del ítem sobre puntaje resto y ola').font = fnt(bold=True, size=10); r += 1
head(ws, r, ['Código', 'Actividad', 'OR ola (DIF uniforme)', 'p uniforme', 'p no uniforme', 'n']); r += 1
for x in P['dif'].itertuples():
    put(ws, r, [x.item, ILAB[x.item], x.or_ola, x.p_unif, x.p_nounif, x.n], {3: '0.00', 4: '0.000', 5: '0.000', 6: '#,##0'}, blue=(3, 4, 5, 6)); r += 1
r += 1; ws.cell(r, 1, 'Correlaciones tetracóricas (conjunto)').font = fnt(bold=True, size=10); r += 1
head(ws, r, [''] + ITEMS); r += 1
T = P['psy']['Conjunto']['tetra']
for k in ITEMS:
    put(ws, r, [k] + list(T.loc[k].values), {j: '0.00' for j in range(2, 8)}, blue=range(2, 8)); r += 1
widths(ws, [10, 40, 11, 11, 11, 12, 12, 12, 9, 9, 9, 9])

# Versiones y cambio
ws = wb.create_sheet('Índice y cambio'); title(ws, 'Cambio del índice entre olas, por versión', 'MCO ponderado (índice ~ ola) con EE robustos HC1, salvo indicación.')
head(ws, 4, ['Versión', 'Descripción', '2023-2024', '2025-2026', 'Diferencia', 'EE', 'IC95 inf', 'IC95 sup', 'p', 'n'])
for i, x in enumerate(R['change'].itertuples(), 5):
    fm = '0.000' if x.version in ('alguna', 'iei_irt') else '0.0'
    put(ws, i, [x.version, x.desc, x.m23, x.m25, f'=D{i}-C{i}', x.se, f'=E{i}-1.96*F{i}', f'=E{i}+1.96*F{i}', x.p, x.n],
        {3: fm, 4: fm, 5: '+' + fm + ';-' + fm, 6: fm, 7: '+' + fm + ';-' + fm, 8: '+' + fm + ';-' + fm, 9: '0.000', 10: '#,##0'}, blue=(3, 4, 6, 9, 10))
i += 2; ws.cell(i, 1, 'Correlaciones entre versiones').font = fnt(bold=True, size=10); i += 1
C = R['corr_idx'] if 'corr_idx' in R else P['corr_idx']
head(ws, i, [''] + list(C.columns)); i += 1
for k in C.index:
    put(ws, i, [k] + list(C.loc[k].values), {j: '0.000' for j in range(2, 7)}, blue=range(2, 7)); i += 1
i += 1; ws.cell(i, 1, 'Distribución del índice principal por tramos').font = fnt(bold=True, size=10); i += 1
head(ws, i, ['Tramo', '2023-2024', '2025-2026', 'Cambio (pp)']); i += 1
Dd = R['dist']
for t in list(dict.fromkeys(Dd.tramo)):
    a = Dd[(Dd.ola == 0) & (Dd.tramo == t)].pct.iloc[0]; b = Dd[(Dd.ola == 1) & (Dd.tramo == t)].pct.iloc[0]
    put(ws, i, [t, a, b, f'=(C{i}-B{i})*100'], {2: '0.0%', 3: '0.0%', 4: '+0.0;-0.0'}, blue=(2, 3)); i += 1
widths(ws, [14, 62, 11, 11, 11, 8, 10, 10, 8, 8])

# Subgrupos
ws = wb.create_sheet('Subgrupos'); title(ws, 'Índice por grupo y ola (0-100)', 'Diferencia por MCO ponderado con EE robustos HC1.')
head(ws, 4, ['Grupo', 'Categoría', '2023-2024', '2025-2026', 'Diferencia', 'EE', 'IC95 inf', 'IC95 sup', 'p', 'Sig.', 'n'])
r = 5; prev = None
for x in R['subg'].itertuples():
    if x.grupo != prev:
        ws.cell(r, 1, x.grupo).font = fnt(bold=True, size=9)
        for j in range(1, 12): ws.cell(r, j).fill = BLK
        r += 1; prev = x.grupo
    put(ws, r, ['', x.categoria, x.m23, x.m25, f'=D{r}-C{r}', x.se, f'=E{r}-1.96*F{r}', f'=E{r}+1.96*F{r}', x.p, star(x.p), x.n],
        {3: '0.0', 4: '0.0', 5: '+0.0;-0.0', 6: '0.0', 7: '+0.0;-0.0', 8: '+0.0;-0.0', 9: '0.000', 11: '#,##0'}, blue=(3, 4, 6, 9, 11)); r += 1
widths(ws, [30, 24, 10, 10, 11, 7, 9, 9, 8, 6, 8]); ws.freeze_panes = 'C5'

# Modelos anidados
ws = wb.create_sheet('Modelos anidados'); title(ws, 'Determinantes del índice: coeficientes MCO ponderados (puntos del índice 0-100)', 'EE robustos HC1 entre paréntesis. *** p<0,001; ** p<0,01; * p<0,05; † p<0,10.')
names = list(R['models']); hs = ['Variable']
for nm in names: hs += [nm, '']
head(ws, 4, hs, 42)
for k in range(len(names)): ws.merge_cells(start_row=4, start_column=2 + 2*k, end_row=4, end_column=3 + 2*k)
r = 5
for b, vs in {'Ola': ['ola25'], **BLOQUES, 'Administración zonal (ref.: Eloy Alfaro)': AZD}.items():
    ws.cell(r, 1, b).font = fnt(bold=True, size=9)
    for j in range(1, 2 + 2*len(names)): ws.cell(r, j).fill = BLK
    r += 1
    for v in vs:
        ws.cell(r, 1, LAB[v]).font = fnt(size=9)
        for k, nm in enumerate(names):
            m = R['models'][nm]
            if v in m['params'].index:
                c = ws.cell(r, 2 + 2*k, m['params'][v]); c.number_format = '+0.00;-0.00'; c.font = fnt(size=9, bold=m['p'][v] < .05)
                c = ws.cell(r, 3 + 2*k, f"({m['bse'][v]:.2f}){star(m['p'][v])}".replace('.', ',')); c.font = fnt(size=8, color='555555')
        r += 1
for lab_, key, fm in [('Constante', 'const', '+0.00;-0.00'), ('Observaciones', 'n', '#,##0'), ('R²', 'r2', '0.000')]:
    ws.cell(r, 1, lab_).font = fnt(bold=True, size=9)
    for k, nm in enumerate(names):
        m = R['models'][nm]; v = m['params']['const'] if key == 'const' else m[key]
        c = ws.cell(r, 2 + 2*k, v); c.font = fnt(size=9); c.number_format = fm
    r += 1
widths(ws, [48] + [9, 12]*len(names)); ws.freeze_panes = 'B5'

# Por ola
ws = wb.create_sheet('Determinantes por ola'); title(ws, '¿Cambiaron los determinantes del índice entre olas?', 'MCO ponderado M4 por ola con EF de AZ. Diferencia = 2025 − 2023.')
head(ws, 4, ['Variable', 'b 2023-2024', 'EE', 'p', 'b 2025-2026', 'EE', 'p', 'Diferencia', 'EE dif.', 'z', 'p dif.', 'Sig. dif.'], 32)
r = 5
for b, vs in {**BLOQUES, 'Administración zonal (ref.: Eloy Alfaro)': AZD}.items():
    ws.cell(r, 1, b).font = fnt(bold=True, size=9)
    for j in range(1, 13): ws.cell(r, j).fill = BLK
    r += 1
    for v in vs:
        e = Bw.loc[v]
        put(ws, r, [LAB[v], e.b_23, e.se_23, e.p_23, e.b_25, e.se_25, e.p_25, f'=E{r}-B{r}', f'=SQRT(C{r}^2+F{r}^2)', f'=H{r}/I{r}',
                    f'=2*(1-NORMSDIST(ABS(J{r})))', f'=IF(K{r}<0.001,"***",IF(K{r}<0.01,"**",IF(K{r}<0.05,"*",IF(K{r}<0.1,"†",""))))'],
            {2: '+0.00;-0.00', 3: '0.00', 4: '0.000', 5: '+0.00;-0.00', 6: '0.00', 7: '0.000', 8: '+0.00;-0.00', 9: '0.00', 10: '0.00', 11: '0.000'}, blue=(2, 3, 4, 5, 6, 7))
        r += 1
ws.conditional_formatting.add(f'A5:L{r-1}', FormulaRule(formula=['AND(ISNUMBER($K5),$K5<0.05)'], font=Font(name=F, bold=True)))
r += 1
for nm, t in [('wald', 'todas las interacciones, incl. AZ'), ('wald_noaz', 'sin efectos fijos de AZ')]:
    ws.cell(r, 1, f"Prueba de Wald conjunta ({t}): χ²({R[nm]['df']}) = {R[nm]['chi2']:.1f}; p = {R[nm]['p']:.2e}").font = fnt(size=9, italic=True); r += 1
widths(ws, [48, 11, 7, 7, 11, 7, 7, 10, 7, 7, 8, 7]); ws.freeze_panes = 'B5'

# Descomposición
ws = wb.create_sheet('Descomposición'); title(ws, 'Descomposición Oaxaca-Blinder del cambio del índice (puntos)', 'Coeficientes del modelo conjunto con indicador de ola. IC 95% bootstrap (300 réplicas).')
head(ws, 4, ['Componente', 'Estimado', 'EE bootstrap', 'IC95 inf', 'IC95 sup', '% del cambio'])
labo = {'total': 'Cambio total (2025 − 2023)', 'explicado': 'Explicado por composición', 'no_explicado': 'No explicado'}
for i, (k, x) in enumerate(ob.iterrows(), 5):
    put(ws, i, [labo.get(k, '   ' + k.replace('E: ', 'Explicado: ')), x.estimado, x.se, x.lo, x.hi, f'=B{i}/$B$5'],
        {2: '+0.00;-0.00', 3: '0.00', 4: '+0.00;-0.00', 5: '+0.00;-0.00', 6: '0.0%'}, blue=(2, 3, 4, 5))
    if k in labo: ws.cell(i, 1).font = fnt(size=9, bold=True)
i += 2; o2 = R['oaxaca_sin_ch6']
ws.cell(i, 1, f"Versión sin CH_6: total {es(o2['total'], 2)}; explicado {es(o2['explicado'], 2)}; no explicado {es(o2['no_explicado'], 2)}.").font = fnt(size=9, italic=True)
widths(ws, [52, 11, 13, 11, 11, 12])

# Sensibilidad
ws = wb.create_sheet('Sensibilidad'); title(ws, 'Efecto de la ola condicional a covariables (M4), según especificación')
head(ws, 4, ['Especificación', 'Coef. ola 2025', 'EE', 'IC95 inf', 'IC95 sup', 'p', 'n'])
for i, x in enumerate(R['sens'].itertuples(), 5):
    fm = '0.000' if 'IRT' in x.especificacion else '0.00'
    put(ws, i, [x.especificacion, x.ola, x.se, f'=B{i}-1.96*C{i}', f'=B{i}+1.96*C{i}', x.p, x.n],
        {2: '+' + fm + ';-' + fm, 3: fm, 4: '+' + fm + ';-' + fm, 5: '+' + fm + ';-' + fm, 6: '0.000', 7: '#,##0'}, blue=(2, 3, 6, 7))
ws.cell(i + 2, 1, 'Las versiones ordinal (0-100) e IRT (desviaciones estándar) están en otra escala que la principal.').font = fnt(size=8, italic=True)
widths(ws, [52, 13, 8, 10, 10, 8, 8])

# Figuras
ws = wb.create_sheet('Figuras'); title(ws, 'Figuras'); pos = 3
for f, h in [('iei_fig1_items.png', 440), ('iei_fig2_cambio.png', 470), ('iei_fig3_determinantes.png', 900)]:
    img = XLImage(f'{SALIDA}/' + f); ratio = img.width / img.height; img.height = h; img.width = int(h * ratio)
    ws.add_image(img, f'A{pos}'); pos += int(h / 20) + 3

# Índice por persona
ws = wb.create_sheet('Índice por persona'); title(ws, 'Índice por persona (para unir a la base por ID_Todo)')
cols = ['ID_Todo', 'AÑO', 'ADMINISTRACION ZONAL', 'peso_az_estrato']
out = d[cols].copy()
for c in ['n_items', 'iei', 'iei_sin_ch6', 'iei_core4', 'iei_ord', 'iei_irt']: out[c] = IDX[c]
head(ws, 3, list(out.columns))
for i, row in enumerate(out.itertuples(index=False), 4):
    for j, v in enumerate(row, 1):
        ws.cell(i, j, None if (isinstance(v, float) and np.isnan(v)) else v)
widths(ws, [12, 7, 20, 14, 8, 9, 11, 10, 9, 9]); ws.freeze_panes = 'A4'
wb.save(f'{SALIDA}/EVPI_indice_evitacion_inseguridad.xlsx'); print('Guardado en', SALIDA)

## 10. Descargar resultados

In [ ]:
import shutil
shutil.make_archive('salidas_iei', 'zip', SALIDA)
if EN_COLAB:
    files.download(f'{SALIDA}/EVPI_indice_evitacion_inseguridad.xlsx')
    # files.download('salidas_iei.zip')   # Excel + figuras